# **Introduction to AI**

## **What is a large language model?**

- One very large mathematical **function** or **mapping**: text in, text out.
- It was fitted once, on a large amount of data from the internet, scanned books, etc. There is a **knowledge
  cutoff** — it has not been trained on anything that appeared after that date.
- The result is a pile of numbers, the **weights**, and they are fixed.
- No database, no internet, no memory of yesterday — unless *you* include it in the context.
- **GPT** = **G**enerative **P**re-trained **T**ransformer: it generates, it was pre-trained once,
  and the architecture is a **transformer** - [Attention is all you need](https://arxiv.org/abs/1706.03762).

## **Tokens and the next token**

<img src="fig_tokens.png" width="950" style="display:block;margin:1.2em auto;">

- The model never sees letters or words. Text is cut into **tokens** — chunks of characters from a
  fixed vocabulary (~10⁵ entries) — and each token is just an integer.
- A common word is one token; rare names, other languages, numbers and code split into several.
- Rule of thumb: ~4 characters ≈ 1 token, 1000 tokens ≈ 750 words.

## **Guessing the next token**

<img src="fig_next_token.png" width="950" style="display:block;margin:1.2em auto;">

- Given the tokens so far, the model returns a **probability for every token in the vocabulary**.
  That is the entire operation: no plan, no goal, no draft of the answer.
- **Hallucination** — a fluent, confident answer that is simply wrong: likely is not the same
  as true.
- **Sampling** then picks one of them; **temperature** says how adventurous that pick is
  (0 = always the most likely token).
- Not all tokens are text. The vocabulary also holds **control tokens** that structure the
  conversation — *start of message*, *end of message*, *end of turn*.

## **...and then it does it again**

<img src="fig_autoregression.png" width="950" style="display:block;margin:1.2em auto;">

- The chosen token is appended to the input and the whole model runs again — **autoregression**.
  A 500-token answer means 500 passes through the whole model.
- Your **input** is processed in one parallel pass, but the **output** has to be produced one token
  at a time. That is why output tokens are typically **3–10× more expensive** than input tokens,
  and why long answers are slow.
- **Reasoning models** first generate a long chain of "thinking" tokens before the visible answer.
  You pay for those too, even though you usually never see them.
- How long it thinks is a dial you set per call — the **effort**: `low` / `medium` / `high` /
  `xhigh`. It has largely taken over from temperature as the knob worth touching.
- High effort pays off for derivations, debugging and planning; on lookups and rewriting it just
  burns tokens and time.

## **Multimodality**

<img src="fig_multimodal.png" width="950" style="display:block;margin:1.2em auto;">

- Images and audio are cut up and encoded into the same sequence of tokens as the text.
- So: paste a plot, a screenshot of a traceback, a photo of the whiteboard.
- Images cost tokens too (~1–2 k for a screenshot). If you want a figure back, usually ask for
  **matplotlib code**, not an image. But some agents have their own visualization tools (e.g. `dataviz` in Claude).

## **How big is a model?**

- A model is a big pile of numbers — its **parameters** (weights).
- To run a model, every parameter has to sit in memory. How much each one costs depends on the
  precision you store it at:
  - **16-bit** (full precision) → 2 bytes per parameter
  - **8-bit** (*quantized*) → 1 byte → the rule of thumb: **~1 GB per 10⁹ parameters**
  - **4-bit** → 0.5 byte
- **Quantization** (fewer bits per parameter) is what makes local models possible at all: 4-bit
  costs a little quality and 4× less memory (when we run models locally).

| model | released | parameters | memory to run it | hardware |
|---|---|---|---|---|
| Llama 3.1 8B | 2024 | 8 B | 16 GB (16-bit) → 5 GB (4-bit) | a laptop |
| Mistral Large 2 | 2024 | 123 B | 246 GB (16-bit) → 70 GB (4-bit) | one 80 GB GPU at 4-bit |
| GPT-3 | 2020 | 175 B | ~350 GB (16-bit) | a few GPUs |
| DeepSeek-V3 | 2024 | 671 B | ~670 GB (8-bit) | an 8-GPU server |
| GPT-4 | 2023 | ~1800 B *(leaked, unconfirmed)* | ~1.8 TB (8-bit) | a rack of GPUs |
| GPT-5, Claude Opus, Gemini | 2025–26 | not published | — | datacenter |

- Bigger ≈ knows more and reasons better, but is slower and more expensive per token. What you can
  run at home is **~100× smaller** than what you rent through an API — that is most of the quality
  gap.

## **Dense vs. Mixture-of-Experts**

<img src="fig_dense_vs_moe.png" width="950" style="display:block;margin:1.2em auto;">

- **Dense** — every parameter runs for every token.
- **MoE** — a router wakes only a few "experts" per layer: **total** parameters ≫ **active**
  parameters (DeepSeek-V3: 671 B total, 37 B active per token).
- Routers pick **several** experts, not one: Mixtral 8x7B takes 2 of 8, DeepSeek-V3 takes 8 of 256
  per layer (plus one shared expert that always runs).
- MoE gives you the knowledge of a huge model at the running cost of a small one — which is why
  "671 B parameters" and "fast and cheap" can be true at the same time.

## **Agent**

<img src="fig_agent_loop.png" width="950" style="display:block;margin:1.2em auto;">

- **Agent = model + tools + a loop**, running until it decides it is done.
- The point is **feedback**: it can run the code, read the error, fix it, run again.
- The loop usually turns 5–50 times before you see one word of the answer.
- So always give it something to check against — a test, a plot, a known limit.

## **Context window**

<img src="fig_context_window.png" width="950" style="display:block;margin:1.2em auto;">

- Every call is **stateless** — a "conversation" is the client re-sending the whole history.
- **Anything not in the context does not exist** for the model: not your last chat, not the
  file you did not open, not yesterday's result.
- Big, but finite: typically **272 K** or **1 M tokens**. 272 K is roughly a 700-page
  book — and a long agent session eats it faster than you would think.
- One way to fill it is **RAG** (Retrieval-Augmented Generation) — search your documents and
  paste the best chunks in. Good for papers and manuals; for code, an agent that can just
  `grep` your repo usually wins.
- Another slice is **tool definitions** — the JSON describing each function the model may ask
  for. It cannot run anything itself: it only emits *"call `run_bash` with `{"command": "ls"}`"*,
  and your code decides whether to obey.
    - **MCP** (**M**odel **C**ontext **P**rotocol) is the open standard for shipping tools as
  reusable servers (lecture 3).
- More is not better: junk in the context measurably degrades answers (*context rot*).
- Choosing what goes in is the real skill — **context engineering**.

A tool definition is just JSON sent along with the prompt:

```json
{"name": "calculate",
 "description": "Evaluate a mathematical expression. Use whenever an exact number is needed.",
 "parameters": {"type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"]}}
```

The description *is* a prompt — badly described tools are badly used. Security is yours, too:
a bash tool runs with your permissions.


## **Memory**

<img src="fig_memory.png" width="950" style="display:block;margin:1.2em auto;">

- Session memory = the transcript; when it overflows it is **compacted** into a summary.
- Long-term memory = a **file** (`AGENTS.md`, notes, your repo) that gets read back in.

## **Harness**

<img src="fig_harness.png" width="950" style="display:block;margin:1.2em auto;">

- You rent the **model**; the **harness** is the program you install around it.
- Codex CLI, Claude Code, Copilot, Cursor, Warp — same picture, different ergonomics.
- The harness decides what the model sees and may do, so it matters as much as the model.
- **Harness vs. agent** — the *agent* is the loop itself (model + tools + loop); the *harness*
  is the program that implements it, wires up the tools and decides what goes into the context.
  Codex CLI is a harness; what it does while it works is an agent.

**More next week - we will install and show Codex.**

## **Glossary**

| term | one line |
|---|---|
| token | chunk of characters the model reads and writes; ~4 characters |
| weights / parameters | the frozen numbers that *are* the model |
| quantization | storing weights in fewer bits (8-bit, 4-bit) so they fit in less memory |
| autoregression | appending each new token to the input and running the model again |
| temperature | how adventurous the sampling is; 0 = most likely token |
| context window | everything the model can see in this one call |
| system prompt | instructions placed at the top of the context |
| reasoning model | trained to think at length, in tokens, before answering |
| effort | how long a reasoning model thinks before it answers |
| dense / MoE | all parameters per token vs. a router picking a few experts |
| multimodal | images and audio encoded as tokens too |
| tool / function call | JSON description + your function; the model only *requests* it |
| MCP | Model Context Protocol, open standard for shipping tools as servers |
| RAG | Retrieval-Augmented Generation; adds documentation into your context |
| agent | model + tools + loop |
| harness | the program that builds the context and runs the loop |

## **[OpenAI playground](https://platform.openai.com/chat)** 

## **OpenAI API**

In [1]:
from openai import OpenAI

# Put your API key here.
# In a real application, use an environment variable or .env file instead.
# API_KEY = "YOUR_API_KEY_HERE"
API_KEY = API_KEY = open("/home/plsek/Documents/Keys/openai_codex.txt").read().strip()

client = OpenAI(api_key=API_KEY)

MODEL = "gpt-5.5"

In [13]:
instructions = "You are a helpful physics tutor. Explain concepts clearly and concisely."
prompt = "What is the difference between a star and a galaxy?"

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=prompt
)

print(response.output_text)

A **star** is a single massive ball of hot gas, mostly hydrogen and helium, that produces light and heat through **nuclear fusion** in its core. The Sun is a star.

A **galaxy** is a huge system containing **billions or even trillions of stars**, along with gas, dust, planets, and dark matter, all held together by gravity. The Milky Way is our galaxy.

So, the difference is:

- **Star:** one object that gives off light  
- **Galaxy:** a vast collection of many stars and other material

For scale: the Sun is one star, and the Milky Way contains over 100 billion stars.


In [9]:
from IPython.display import display, Markdown

instructions="You are a helpful physics tutor. Explain concepts clearly and concisely."
prompt = "What is the difference between a star and a galaxy?"

stream = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=prompt,
    stream=True
)

output = display(Markdown(""), display_id=True)
text = ""

for event in stream:
    if event.type == "response.output_text.delta":
        text += event.delta
        output.update(Markdown(text))

A **star** is a single huge ball of hot gas/plasma that produces light and heat through nuclear fusion.  
Example: **the Sun**.

A **galaxy** is a massive collection of many things held together by gravity, including **billions of stars**, gas, dust, planets, and dark matter.  
Example: **the Milky Way**, which contains our Sun.

So the key difference is:

- **Star:** one glowing object  
- **Galaxy:** a huge system containing many stars

In short: **a star can be part of a galaxy, but a galaxy contains billions of stars.**

## **Reasoning**

In [10]:
from IPython.display import display, Markdown

instructions = "You are a physics tutor. Solve problems carefully and explain the important steps."
prompt = "A galaxy cluster has a total mass of 5e14 solar masses. Estimate its characteristic virial velocity assuming a radius of 1 Mpc."

stream = client.responses.create(
    model="gpt-5.5",
    instructions=instructions,
    input=prompt,
    reasoning={"effort": "xhigh",
               "summary": "detailed"},
    stream=True)


reasoning = ""
answer = ""

reasoning_display = display(Markdown("### Reasoning\n"), display_id=True)
answer_display = display(Markdown("### Final answer\n"), display_id=True)

for event in stream:
    # Reasoning summary
    if event.type == "response.reasoning_summary_text.delta":
        reasoning += event.delta
        reasoning_display.update(Markdown("### Reasoning\n\n" + reasoning))

    # Final answer
    elif event.type == "response.output_text.delta":
        answer += event.delta
        answer_display.update(Markdown("### Final answer\n\n" + answer))

### Reasoning

**Estimating virial velocity**

I need to estimate the virial velocity of a galaxy cluster with a mass of 5e14 solar masses and a radius of 1 Mpc. I'll start using the virial theorem, which leads to the formula v  sqrt(GM/R). First, I need to convert the mass and radius into the correct units. 

After calculating, I find a characteristic virial velocity to be about 1466 km/s, and if I consider a factor of sqrt(1/2), it would drop to around 1036 km/s. Finally, it seems the virial velocity could be approximately 1500 km/s.**Calculating characteristic virial velocity**

The question asks for the characteristic virial velocity, given a specific radius. I can estimate this to be around 1.5e3 km/s, factoring in unity factors. I might consider using astrophysical units, like G = 4.302e-6 kpc (km/s)² Msun⁻¹ with R = 1000 kpc. By calculating v as sqrt(4.302e-6 * 5e14/1000), I simplify it to about 1467 km/s. 

I should also explain that using the virial theorem, 2K + U = 0, leads to the relationship v²  GM/R.

### Final answer

Using the virial estimate,

\[
v_{\rm vir} \sim \sqrt{\frac{GM}{R}}
\]

with

- \(M = 5\times 10^{14} M_\odot\)
- \(R = 1\ \text{Mpc} = 1000\ \text{kpc}\)
- \(G = 4.302\times 10^{-6}\ \frac{\text{kpc}\,(\text{km/s})^2}{M_\odot}\)

we get

\[
v_{\rm vir} \sim \sqrt{
\frac{(4.302\times 10^{-6})(5\times 10^{14})}{1000}
}
\]

\[
v_{\rm vir} \sim \sqrt{2.15\times 10^6}\ \text{km/s}
\]

\[
v_{\rm vir} \approx 1.5\times 10^3\ \text{km/s}
\]

So the characteristic virial velocity is approximately

\[
\boxed{v_{\rm vir} \sim 1500\ \text{km/s}}
\]

up to factors of order unity depending on the exact virial definition.

## **Simple chatbot**

In [11]:
conversation = []

while True:
    user_input = input("You: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    conversation.append({
        "role": "user",
        "content": user_input
    })

    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful and concise chatbot.",
        input=conversation
    )

    print("Assistant:", response.output_text)

    conversation.append({
        "role": "assistant",
        "content": response.output_text
    })

You:  exit


## **Simple AI agent**

                 ┌──────────────┐
                 │     User     │
                 └──────┬───────┘
                        │
                        ▼
                 ┌──────────────┐
                 │     LLM      │
                 └──────┬───────┘
                        │
              decides what to do
                        │
              ┌─────────┴─────────┐──────────────────┐
              │                   │                  │
              ▼                   ▼                  │
       ┌──────────────┐    ┌──────────────┐          │
       │  Calculator  │    │     Bash     │          │
       │              │    │     Tool     │          │
       └──────┬───────┘    └──────┬───────┘          │
              │                   │                  │
              └─────────┬─────────┘                  │ 
                        │                            │
                        ▼                            │
                 tool results                        │
                        │                    answer directly
                        ▼                            │
                 ┌──────────────┐                    │
                 │     LLM      │                    │
                 └──────┬───────┘                    │
                        │                            │
               need another action?                  │
                        │                            │
              ┌─────────┴─────────┐                  │
              │                   │                  │
              │ yes               │ no               │
              │                   │                  │
              │                   ▼                  │
              ▼            ┌──────────────┐          │
      another tool call    │ final answer │──────────┘
                           └──────────────┘

In [2]:
import json
import math
import subprocess
from IPython.display import display, Markdown


# Calculator tool
def calculate(expression):
    return eval(
        expression,
        {
            "__builtins__": {},
            "sqrt": math.sqrt,
            "pi": math.pi,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "exp": math.exp,
        },
        {}
    )

# Bash tool
def run_bash(command):
    """
    Execute a bash command and return its output.
    """

    result = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True,
        timeout=10)

    output = ""
    if result.stdout:
        output += result.stdout
    if result.stderr:
        output += "\n[stderr]\n" + result.stderr
    output += f"\n[exit code: {result.returncode}]"

    return output

# Tools available to the agent
tools = [
    {
        "type": "function",
        "name": "calculate",
        "description": (
            "Evaluate a mathematical expression. "
            "Available functions and constants include "
            "sqrt, pi, sin, cos, tan, log, and exp."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Mathematical expression to evaluate."
                }
            },
            "required": ["expression"]
        }
    },
    {
        "type": "function",
        "name": "run_bash",
        "description": (
            "Execute a bash command on the local computer and return "
            "its output. Use this when you need to inspect the local "
            "environment or work with files."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "Bash command to execute."
                }
            },
            "required": ["command"]
        }
    }
]

# Execute tools
def execute_tool(name, arguments):
    if name == "calculate":
        return calculate(arguments["expression"])

    if name == "run_bash":
        return run_bash(arguments["command"])

    raise ValueError(f"Unknown tool: {name}")

# Markdown / LaTeX formatting
def format_answer(text):
    text = text.replace(r"\(", "$")
    text = text.replace(r"\)", "$")
    text = text.replace(r"\[", "$$")
    text = text.replace(r"\]", "$$")
    return text

# Display helpers
def show_reasoning(text):
    display(Markdown(f"\n**Reasoning:**\n {format_answer(text)}"))

def show_tool_call(name, arguments):
    display(Markdown(
            f"> **Tool call:** `{name}`  \n"
            f"> **Arguments:** `{arguments}`"))

# Stream final answer
def stream_final_answer(response):
    text = ""
    display_handle = display(
        Markdown("\n**Assistant:**"),
        display_id=True)

    for event in response:
        if event.type == "response.output_text.delta":
            text += event.delta
            display_handle.update(
                Markdown(f"\n**Assistant:**\n {format_answer(text)}"))
    return text


# Agent
previous_response_id = None

def run_agent(user_input):
    global previous_response_id

    response = client.responses.create(
        model=MODEL,
        instructions=(
            "You are a helpful scientific assistant and autonomous agent. "
            "You have access to a calculator and a bash command tool. "
            "Use the calculator whenever an accurate numerical calculation "
            "is required. "
            "Use bash when you need to inspect the local computer, files, "
            "directories, or environment. "
            "Think carefully before using tools. "
            "Do not execute destructive commands unless explicitly asked. "
            "Format your final answers using Markdown and LaTeX where "
            "appropriate."
        ),
        input=user_input,
        previous_response_id=previous_response_id,
        tools=tools,
        reasoning={"effort": "medium",
                   "summary": "auto"})

    while True:
        tool_outputs = []
        for item in response.output:
            # Reasoning summary
            if item.type == "reasoning":
                if item.summary:
                    summary_text = ""
                    for summary in item.summary:
                        if summary.type == "summary_text":
                            summary_text += summary.text
                    if summary_text:
                        show_reasoning(summary_text)

            # Tool call
            elif item.type == "function_call":
                arguments = json.loads(item.arguments)
                show_tool_call(
                    item.name,
                    arguments
                )
                result = execute_tool(
                    item.name,
                    arguments
                )
                tool_outputs.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": str(result)
                })

        # No tool calls -> generate final answer
        if not tool_outputs:
            previous_response_id = response.id
            stream = client.responses.create(
                model=MODEL,
                previous_response_id=response.id,
                input=[],
                tools=tools,
                stream=True
            )
            return stream_final_answer(stream)

        # Send tool results back to the model
        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
            reasoning={"effort": "medium",
                       "summary": "auto"})


# Interactive agent
while True:
    display(Markdown("---"))
    user_input = input("User: ")
    print()
    if user_input.lower() in {"exit", "quit"}:
        break

    run_agent(user_input)

---

User:  "A galaxy cluster has a total mass of 5e14 solar masses. Estimate its characteristic virial velocity assuming a radius of 1 Mpc."



**Reasoning:**
 **Calculating velocity**

I need to answer simply by computing velocity with the formula v  sqrt(GM/R). I need to use appropriate units for G, which is G = 4.30091e-6 (km/s)^2 kpc/Msun. For M, I'm using 5e14 Msun, and R is 1 Mpc, or 1000 kpc. So, I get v = sqrt(4.30091e-6 * 5e14 / 1000) km/s, simplifying down to about 1466 km/s. It's interesting to think about virial velocity, which could also depend on context with values around 1000-1500 km/s.

> **Tool call:** `calculate`  
> **Arguments:** `{'expression': 'sqrt(4.30091e-6 * 5e14 / 1000)'}`


**Assistant:**
 Using the virial estimate,

$$
v \sim \sqrt{\frac{GM}{R}}
$$

with

$$
G = 4.3\times 10^{-6}\ \frac{\mathrm{kpc}\,(\mathrm{km/s})^2}{M_\odot},
$$

$$
M = 5\times 10^{14} M_\odot,
$$

and

$$
R = 1\ \mathrm{Mpc} = 1000\ \mathrm{kpc},
$$

we get

$$
v \sim \sqrt{\frac{(4.3\times 10^{-6})(5\times 10^{14})}{1000}}
$$

$$
v \approx 1.5\times 10^3\ \mathrm{km/s}.
$$

So the characteristic virial velocity is approximately

$$
\boxed{v \sim 1500\ \mathrm{km/s}}.
$$

---

User:  exit


In [3]:
# A galaxy cluster has a total mass of 5e14 solar masses. Estimate its characteristic virial velocity assuming a radius of 1 Mpc.
# Turn this into a python script that plots the velocity as a function of radius.
# Now add a slider that controls that mass of the system and updates the plot.